# HGT experiment: random context order

Ноутбук для проверки порядка добавления контекстных признаков.

In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd()))

from hgt_common import *

DATA_DIR = Path("..")
CFG = HGTExperimentConfig(
    k_values=[20],
    target_metrics=["recall", "ndcg"],
    epochs=100,
)
set_seed(CFG.seed)
log_step(f"device: {CFG.device}")
log_step(f"target k_values: {CFG.k_values}")
log_step(f"target metrics: {CFG.target_metrics}")

tables = load_hgt_tables(DATA_DIR)
runner = make_hgt_runner(tables, CFG)


[00:30:29] device: cpu
[00:30:29] target k_values: [20]
[00:30:29] target metrics: ['recall', 'ndcg']


/Users/alexandro/DataspellProjects/recommend-system/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[00:30:29] loaded rates: (948367, 4)
[00:30:29] loaded users: (6040, 4)
[00:30:29] loaded movies: (3433, 7)
[00:30:29] loaded movie_genres: (6408, 2)
[00:30:29] loaded movie_actors: (10285, 2)
[00:30:29] loaded movie_directors: (3676, 2)
[00:30:29] loaded movie_countries: (4765, 2)
[00:30:29] loaded movie_tags: (450912, 2)


## Случайные порядки добавления контекста

Этот сценарий нужен, если хочется проверить зависимость результата от порядка добавления признаков. Для чистого вклада отдельного признака используй `hgt_exp_single_context.ipynb`.

In [2]:
import random
from pathlib import Path

import pandas as pd

CONTEXT_GROUPS = [
    "genres",
    "directors",
    "actors",
    "countries"
]

RESULTS_PATH = Path("hgt_random_context_recall20_ndcg20_results.csv")
SUMMARY_PATH = Path("hgt_random_context_recall20_ndcg20_summary.csv")


def make_random_context_orders(context_groups, n_runs=5, random_state=42):
    rng = random.Random(random_state)
    orders = {}

    for run_id in range(1, n_runs + 1):
        order = list(context_groups)
        rng.shuffle(order)
        orders[run_id] = order

    return orders


def stage_name_from_contexts(active_contexts):
    if not active_contexts:
        return "raw"
    return "_".join(active_contexts)


def load_existing_results(path=RESULTS_PATH):
    columns = [
        "run_id",
        "step_id",
        "order",
        "stage",
        "added_context",
        "previous_stage",
        "k",
        "metric",
        "value",
        "delta",
    ]

    if path.exists():
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(columns=columns)
        return df

    return pd.DataFrame(columns=columns)


def save_results_incrementally(new_rows, path=RESULTS_PATH):
    new_df = pd.DataFrame(new_rows)

    if path.exists():
        old_df = pd.read_csv(path)
        result_df = pd.concat([old_df, new_df], ignore_index=True)
        result_df = result_df.drop_duplicates(
            subset=["run_id", "step_id", "stage", "metric"],
            keep="last",
        )
    else:
        result_df = new_df

    result_df = result_df.sort_values(["run_id", "step_id", "metric"]).reset_index(drop=True)
    result_df.to_csv(path, index=False)
    return result_df


def summarize_context_contribution(results_df):
    contrib_df = results_df[
        (results_df["added_context"] != "raw") &
        (results_df["delta"].notna())
    ].copy()

    if contrib_df.empty:
        return pd.DataFrame()

    summary = (
        contrib_df
        .groupby(["metric", "added_context"])["delta"]
        .agg(
            mean_delta="mean",
            std_delta="std",
            min_delta="min",
            max_delta="max",
            n_runs="count",
        )
        .reset_index()
    )

    positive_share = (
        contrib_df
        .assign(is_positive=contrib_df["delta"] > 0)
        .groupby(["metric", "added_context"])["is_positive"]
        .mean()
        .reset_index(name="positive_share")
    )

    summary = summary.merge(positive_share, on=["metric", "added_context"], how="left")
    summary = summary.sort_values(["metric", "mean_delta"], ascending=[True, False]).reset_index(drop=True)

    return summary


def update_summary(path=RESULTS_PATH, summary_path=SUMMARY_PATH):
    results_df = load_existing_results(path)

    if results_df.empty:
        print("Пока нет сохранённых результатов.")
        return pd.DataFrame()

    summary_df = summarize_context_contribution(results_df)
    summary_df.to_csv(summary_path, index=False)

    return summary_df


def run_single_random_context_run(
        runner,
        run_id,
        n_runs=5,
        random_state=42,
        threshold=5.0,
        min_pos=5,
        target_k=20,
        target_metrics=("recall", "ndcg"),
        results_path=RESULTS_PATH,
):
    orders = make_random_context_orders(
        CONTEXT_GROUPS,
        n_runs=n_runs,
        random_state=random_state,
    )

    if run_id not in orders:
        raise ValueError(f"run_id must be in [1, {n_runs}]")

    order = orders[run_id]
    existing_df = load_existing_results(results_path)

    completed_steps = set()
    if not existing_df.empty and "run_id" in existing_df.columns:
        completed_steps = set(
            existing_df.loc[existing_df["run_id"] == run_id, "step_id"].astype(int).tolist()
        )

    print("\n" + "=" * 80)
    print(f"RUN {run_id}/{n_runs}")
    print("Порядок добавления:", " -> ".join(order))
    print("Уже выполненные шаги:", sorted(completed_steps))
    print("=" * 80)

    active_contexts = []
    new_rows = []

    previous_values = {metric: None for metric in target_metrics}
    previous_stage = None

    if not existing_df.empty and "run_id" in existing_df.columns:
        saved_run_df = existing_df[existing_df["run_id"] == run_id].copy()
    else:
        saved_run_df = pd.DataFrame(columns=existing_df.columns)

    if not saved_run_df.empty:
        saved_run_df = saved_run_df.sort_values(["step_id", "metric"])

    stage_sequence = [None] + order

    for step_id, added_context in enumerate(stage_sequence):
        if added_context is not None:
            active_contexts.append(added_context)

        stage = stage_name_from_contexts(active_contexts)

        if step_id in completed_steps:
            saved_step = saved_run_df[saved_run_df["step_id"] == step_id]
            for metric in target_metrics:
                metric_rows = saved_step[saved_step["metric"] == metric]
                if not metric_rows.empty:
                    previous_values[metric] = float(metric_rows.iloc[-1]["value"])
            previous_stage = stage
            print(f"SKIP run={run_id}, step={step_id}, stage={stage} уже сохранён")
            continue

        print("\n" + "-" * 80)
        print(f"START run={run_id}, step={step_id}, stage={stage}")
        print("-" * 80)

        result_df, _ = runner.run_temporal_80_20(
            stage=stage,
            threshold=threshold,
            min_pos=min_pos,
            k_values=[target_k],
        )

        metric_row = result_df[result_df["k"] == target_k].iloc[0]
        rows_for_step = []

        for metric in target_metrics:
            current_value = float(metric_row[metric])
            previous_value = previous_values.get(metric)
            delta = None if previous_value is None else current_value - previous_value

            rows_for_step.append({
                "run_id": run_id,
                "step_id": step_id,
                "order": " -> ".join(order),
                "stage": stage,
                "added_context": "raw" if added_context is None else added_context,
                "previous_stage": previous_stage,
                "k": target_k,
                "metric": metric,
                "value": current_value,
                "delta": delta,
            })

            previous_values[metric] = current_value

        save_results_incrementally(rows_for_step, results_path)
        update_summary(results_path)
        new_rows.extend(rows_for_step)

        previous_stage = stage
        print("SAVED", rows_for_step)

    return load_existing_results(results_path)



In [3]:
random_context_results = run_single_random_context_run(
    runner=runner,
    run_id=1,
    n_runs=5,
    random_state=42,
    threshold=5.0,
    min_pos=5,
    target_k=20,
    target_metrics=("recall", "ndcg"),
)

context_contribution_summary = update_summary()

display(random_context_results)
display(context_contribution_summary)


RUN 1/5
Порядок добавления: actors -> directors -> countries -> genres
Уже выполненные шаги: []

--------------------------------------------------------------------------------
START run=1, step=0, stage=raw
--------------------------------------------------------------------------------
[00:30:29] stage=raw: preparing temporal 80/20 split; threshold=5.0, min_pos=5
[00:30:29] stage=raw: positive stats
users_total    5614.0
min               5.0
p25              13.0
median           24.0
p75              48.0
max             533.0
users_lt5         0.0
users_lt10      868.0
users_lt20     2332.0
dtype: float64
[00:30:29] stage=raw: train/test shapes: train=(168107, 3), test=(44791, 3)
[00:30:29] stage=raw: fit started; k_values=[20]; target_metrics=['recall', 'ndcg']
[00:30:29] stage=raw: building index maps
[00:30:29] context: filtering context tables by train movies
[00:30:29] context sizes after filtering: genres=4908, directors=1747, actors=3675, countries=3952, tags=209972
[00:3

KeyboardInterrupt: 